# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnkhamis11/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

### 1. My Lane Selection
**Lane:** Search Intelligence & Organic Visibility (Rank Retention & Engagement Proxy)

**Why this lane:**
Search performance data from Google Search Console and Analytics contains high-volume, noisy time series. Rather than attempting to reverse-engineer Google's proprietary search algorithm, this lane focuses on quantifying client search query stability and high-intent click engagement. Building an end-to-end predictive pipeline for search query performance provides immediate decision support for content, technical SEO, and digital strategy teams.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verification Code: Imports and setup check
import sys
import pandas as pd
import numpy as np

print(f"Python Version: {sys.version.split()[0]}")
print(f"Pandas Version: {pd.__version__}")
print("Environment ready for ML-02 execution.")

Python Version: 3.12.13
Pandas Version: 2.2.2
Environment ready for ML-02 execution.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*
### 2. Decision Framework
* **Decision Improved:** Identifying which high-volume client search queries are at imminent risk of organic rank decay or engagement drops before severe organic traffic loss occurs.
* **Who Acts On It:** SEO Strategists, Content Leads, and Technical Marketing Engineers.
* **Action Taken:** Proactive content updates, internal link restructuring, and technical search optimizations targeted directly at high-risk query clusters.
* **Cost of a Wrong Call:**
  * *False Positive (Predicting decay when stable):* Wasted engineering and editorial resources spent optimizing content that already performs well.
  * *False Negative (Missing an actual decay):* Loss of high-intent organic traffic to competitors, leading to diminished conversions and higher customer acquisition costs (CAC) through paid search alternatives.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Simulation of decision cost structure
unit_cost_false_positive = 150  # Estimated wasted editor hours/cost ($)
unit_cost_false_negative = 1200 # Estimated lost organic revenue ($)

print("Decision Impact Framework initialized:")
print(f"Relative risk ratio (FN/FP): {unit_cost_false_negative / unit_cost_false_positive:.1f}x higher cost for missed decays.")

Decision Impact Framework initialized:
Relative risk ratio (FN/FP): 8.0x higher cost for missed decays.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*
### 3. Data Exploration & Summary Metrics
Using DuckDB with Hugging Face HTTP Bearer authentication to query the March 2026 warehouse slice (`month=2026-03`). We calculate three foundational data metrics:
1. Total observation count (daily client-query pairs).
2. Overall query engagement rate (`clicks > 0`).
3. Average historic impression volume across unique search queries.

In [14]:
import duckdb
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download

# 1. Retrieve Hugging Face token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# 2. Inspect the repository tree to locate the exact Parquet filename
api = HfApi(token=hf_token)
repo_files = api.list_repo_files(repo_id="FlyRank/internship-warehouse", repo_type="dataset")

# Filter files under month=2026-03
target_files = [f for f in repo_files if "month=2026-03" in f and f.endswith(".parquet")]

if not target_files:
    target_files = [f for f in repo_files if f.endswith(".parquet")]

print(f"Found dataset file: {target_files[0]}")

# 3. Download the resolved path to local runtime memory
local_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename=target_files[0],
    repo_type="dataset",
    token=hf_token
)

# 4. Execute query using DuckDB with corrected column names
con = duckdb.connect()

stats_df = con.execute(f"""
SELECT
    COUNT(*) as total_rows,
    ROUND(AVG(CASE WHEN gsc_clicks > 0 THEN 1.0 ELSE 0.0 END) * 100, 2) as engagement_rate_pct,
    ROUND(AVG(gsc_impressions), 2) as avg_impressions_per_query
FROM read_parquet('{local_file}');
""").df()

print("\n--- Real Data Metrics (Month: 2026-03) ---")
print(f"1. Total Dataset Rows: {stats_df['total_rows'].iloc[0]:,}")
print(f"2. Engagement Rate (Clicks > 0): {stats_df['engagement_rate_pct'].iloc[0]}%")
print(f"3. Mean Impressions / Query: {stats_df['avg_impressions_per_query'].iloc[0]}")

Found dataset file: fact_content_daily_performance/month=2026-03/data_0.parquet

--- Real Data Metrics (Month: 2026-03) ---
1. Total Dataset Rows: 9,841,378
2. Engagement Rate (Clicks > 0): 4.25%
3. Mean Impressions / Query: 28.52


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*
### 4. Scope and Claim Boundaries

**What this work CAN claim:**
* **Observed Metrics:** Historically measured query positions, click frequencies, and aggregated engagement trends across observed client domains.
* **Directional Signals:** Directional shifts in query visibility probability based on historical feature aggregations.
* **Decision Support:** Risk prioritization models that assist marketing teams in allocating optimization resources effectively.

**What this work CANNOT and WILL NOT claim:**
* **Causal Proof:** We cannot claim that a specific algorithm update directly caused a rank shift (correlation $\neq$ causation).
* **Predicting Google:** We do not claim to reverse-engineer or predict Google's search algorithm.
* **Deterministic Guarantees:** Predictions are probabilistic decision-support estimates, not guaranteed ranking promises.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Validation of boundary principles in code outputs
claim_boundaries = {
    "Allowed": ["observed", "measured", "directional", "decision-support"],
    "Prohibited": ["causal proof", "predicting Google", "guaranteed rank"]
}

print("Scope boundaries validated:")
for category, terms in claim_boundaries.items():
    print(f"  {category} terms: {', '.join(terms)}")

Scope boundaries validated:
  Allowed terms: observed, measured, directional, decision-support
  Prohibited terms: causal proof, predicting Google, guaranteed rank


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.